In [1]:
# -------------------------------------------------------
# CELL 1: Data Agent Simulation — NL to KQL Pattern
# Purpose: Demonstrate natural language to query translation
#          pattern used by Fabric Data Agent
# Note: Fabric Data Agent requires F64+ SKU. This notebook
#       simulates the NL->KQL->Result pattern manually.
# -------------------------------------------------------

# Sample questions a Data Agent would answer
nl_questions = [
    "Which country has the highest average PM2.5?",
    "How many hazardous readings were detected?",
    "Which stations exceeded WHO guidelines?"
]

# Equivalent KQL/SQL queries for each question
queries = {
    "Which country has the highest average PM2.5?": """
        SELECT country_sk, AVG(value) as avg_pm25
        FROM fact_readings
        WHERE parameter = 'pm25'
        GROUP BY country_sk
        ORDER BY avg_pm25 DESC
        LIMIT 1
    """,
    "How many hazardous readings were detected?": """
        SELECT COUNT(*) as hazardous_count
        FROM fact_readings
        WHERE aqi_category = 'Hazardous'
    """,
    "Which stations exceeded WHO guidelines?": """
        SELECT DISTINCT location_id, parameter, value
        FROM fact_readings
        WHERE exceeds_who_guideline = true
        ORDER BY value DESC
    """
}

print("=== Data Agent — NL to SQL Pattern ===\n")
for question in nl_questions:
    print(f"Q: {question}")
    print(f"SQL: {queries[question].strip()}")
    print()

StatementMeta(, 926e075d-e724-452f-b389-763ed6d22d36, 3, Finished, Available, Finished, False)

=== Data Agent — NL to SQL Pattern ===

Q: Which country has the highest average PM2.5?
SQL: SELECT country_sk, AVG(value) as avg_pm25
        FROM fact_readings
        WHERE parameter = 'pm25'
        GROUP BY country_sk
        ORDER BY avg_pm25 DESC
        LIMIT 1

Q: How many hazardous readings were detected?
SQL: SELECT COUNT(*) as hazardous_count
        FROM fact_readings
        WHERE aqi_category = 'Hazardous'

Q: Which stations exceeded WHO guidelines?
SQL: SELECT DISTINCT location_id, parameter, value
        FROM fact_readings
        WHERE exceeds_who_guideline = true
        ORDER BY value DESC



In [2]:
# -------------------------------------------------------
# CELL 2: Execute NL-derived queries against Gold lakehouse
# Purpose: Show end-to-end NL->Query->Result flow
# -------------------------------------------------------

print("=== Q1: Which country has the highest average PM2.5? ===")
spark.sql("""
    SELECT country_sk, ROUND(AVG(value), 2) as avg_pm25
    FROM fact_readings
    WHERE parameter = 'pm25'
    GROUP BY country_sk
    ORDER BY avg_pm25 DESC
    LIMIT 5
""").show()

print("=== Q2: How many hazardous readings were detected? ===")
spark.sql("""
    SELECT COUNT(*) as hazardous_count
    FROM fact_readings
    WHERE aqi_category = 'Hazardous'
""").show()

print("=== Q3: Which stations exceeded WHO guidelines? ===")
spark.sql("""
    SELECT DISTINCT location_id, parameter, ROUND(value, 2) as value
    FROM fact_readings
    WHERE exceeds_who_guideline = true
    ORDER BY value DESC
    LIMIT 10
""").show()

StatementMeta(, 926e075d-e724-452f-b389-763ed6d22d36, 4, Finished, Available, Finished, False)

=== Q1: Which country has the highest average PM2.5? ===
+-----------+--------+
| country_sk|avg_pm25|
+-----------+--------+
| -251229660|  175.83|
|-1788195040|   131.4|
|  199844526|    35.0|
| -418823411|    20.0|
|-2079833584|    14.3|
+-----------+--------+

=== Q2: How many hazardous readings were detected? ===
+---------------+
|hazardous_count|
+---------------+
|              5|
+---------------+

=== Q3: Which stations exceeded WHO guidelines? ===
+-----------+---------+------+
|location_id|parameter| value|
+-----------+---------+------+
|         30|       co|8720.0|
|         69|       co|8610.0|
|         17|       co|6850.0|
|         30|     pm10| 931.0|
|         17|     pm10| 582.0|
|        103|     pm10| 345.0|
|         50|     pm10| 308.0|
|         13|     pm25| 300.0|
|        103|     pm25| 219.0|
|         46|     pm25| 207.0|
+-----------+---------+------+

